In [1]:
%pip install duckdb pydeck pandas

Note: you may need to restart the kernel to use updated packages.


In [22]:
import duckdb
import pydeck as pdk
import numpy as np
import os
import pandas as pd

In [ ]:
path = "/mnt/shared_data/finflow/gfw_clean"

first_file = os.listdir(path)[0]
df = pd.read_parquet(f"{path}/{first_file}")

print(df.info())
df.head()

In [27]:
# 1. DuckDB Setup & H3 Extension
con = duckdb.connect()
con.execute("INSTALL h3 FROM community;")
con.execute("LOAD h3;")

# Konfiguration
RESOLUTION = 4
PARQUET_PATH = '/mnt/shared_data/finflow/gfw_clean/*.parquet'

# 2. Aggregation der Fischereidaten
query = f"""
    SELECT 
        h3_h3_to_string(
            h3_cell_to_parent(h3_string_to_h3(h3_index), {RESOLUTION})
        ) as h3, 
        SUM(fishing_hours) as val
    FROM read_parquet('{PARQUET_PATH}')
    GROUP BY 1
    HAVING val > 0
"""

print(f"Aggregiere global auf H3-Resolution {RESOLUTION}...")
df = con.execute(query).df()
print(f"Erfolg! {len(df):,} Hexagone aggregiert.")

# 3. Logarithmische Skalierung für Farbe und Höhe
df['log_val'] = np.log1p(df['val'])
max_log = df['log_val'].max()

# 4. Pydeck Layer Definition
# Kontrast-Logik: (ratio * ratio * ratio) für einen steilen Anstieg bei Hotspots
layer = pdk.Layer(
    "H3HexagonLayer",
    df,
    get_hexagon="h3",
    # Farbe: Dunkelrot (40) bis Hellrot (255) mit hohem Kontrast (hoch 3)
    get_fill_color=f"""[
        40 + ((log_val / {max_log}) * (log_val / {max_log}) * (log_val / {max_log})) * 215, 
        0, 
        0, 
        200
    ]""",
    # Höhe: Log-Wert für bessere Sichtbarkeit kleinerer Aktivitäten
    get_elevation="log_val",
    # 30.000 - 50.000 ist ideal für Resolution 4
    elevation_scale=40000, 
    extruded=True,
    pickable=True,
)

# 5. Kamera-Einstellungen
view_state = pdk.ViewState(
    latitude=15, 
    longitude=0, 
    zoom=2.2, 
    pitch=45,
    bearing=0
)

# 6. Deck erstellen & als HTML speichern
r = pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state,
    # Freie dunkle Basiskarte (CartoDB)
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
    tooltip={
        "html": "<b>H3 Index:</b> {h3}<br/><b>Fishing Hours:</b> {val}",
        "style": {"color": "white", "backgroundColor": "#111"}
    }
)

# Speicherpfad
output_file = "views/gfw.html"
r.to_html(output_file)

print(f"Die Karte wurde erfolgreich unter '{output_file}' erstellt!")

Aggregiere global auf H3-Resolution 4...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Erfolg! 189,075 Hexagone aggregiert.
Die Karte wurde erfolgreich unter 'views/gfw.html' erstellt!


In [23]:
path = "/mnt/shared_data/finflow/obis_clean"

first_file = os.listdir(path)[0]
df = pd.read_parquet(f"{path}/{first_file}")

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   decimalLatitude   70000 non-null  float64
 1   decimalLongitude  70000 non-null  float64
 2   best_time_marker  70000 non-null  object 
 3   scientificName    70000 non-null  object 
 4   class             70000 non-null  object 
 5   individualCount   10142 non-null  object 
 6   h3_index          70000 non-null  object 
dtypes: float64(2), object(5)
memory usage: 3.7+ MB
None


,decimalLatitude,decimalLongitude,best_time_marker,scientificName,class,individualCount,h3_index
0,49.1974,-4.1021,2011-03-16 09:59:00+00,Scyliorhinus canicula,Elasmobranchii,None,871871d45ffffff
1,50.1330,-3.5263,2008-10-15 18:54:00+00,Scyliorhinus canicula,Elasmobranchii,None,871875d6bffffff
2,55.6450,-8.2670,2018-11-02 16:56:00+00,Scyliorhinus canicula,Elasmobranchii,None,87182404affffff
3,49.1295,-1.8665,2020-06-24 06:19:00+00,Scyliorhinus canicula,Elasmobranchii,None,871860495ffffff
4,55.7610,-8.0470,2016-10-04 08:49:00+00,Scyliorhinus canicula,Elasmobranchii,None,871824ab1ffffff


In [28]:
# 1. DuckDB Setup & H3 Extension
con = duckdb.connect()
con.execute("INSTALL h3 FROM community;")
con.execute("LOAD h3;")

# Konfiguration
RESOLUTION = 4
OBIS_PATH = '/mnt/shared_data/finflow/obis_clean/*.parquet'

# 2. Aggregation der OBIS-Tierdaten
# Wir nutzen COUNT(*), da jede Zeile in OBIS eine Sichtung darstellt
query_obis = f"""
    SELECT 
        h3_h3_to_string(
            h3_cell_to_parent(h3_string_to_h3(h3_index), {RESOLUTION})
        ) as h3, 
        COUNT(*) as sighting_count
    FROM read_parquet('{OBIS_PATH}')
    GROUP BY 1
    HAVING sighting_count > 0
"""

print(f"Aggregiere OBIS-Daten auf H3-Resolution {RESOLUTION}...")
df_obis = con.execute(query_obis).df()
print(f"Erfolg! {len(df_obis):,} Hexagone mit Tiersichtungen aggregiert.")

# 3. Logarithmische Skalierung
df_obis['log_count'] = np.log1p(df_obis['sighting_count'])
max_log_obis = df_obis['log_count'].max()

# 4. Pydeck Layer Definition (Türkis/Cyan)
# Wir nutzen (ratio * ratio * ratio) für maximalen Kontrast
layer_obis = pdk.Layer(
    "H3HexagonLayer",
    df_obis,
    get_hexagon="h3",
    # Farbe: Türkis [0, G, B, Alpha]
    # Grün und Blau steigen synchron an, um helles Türkis bei Hotspots zu erzeugen
    get_fill_color=f"""[
        0, 
        40 + ((log_count / {max_log_obis}) * (log_count / {max_log_obis}) * (log_count / {max_log_obis})) * 215, 
        40 + ((log_count / {max_log_obis}) * (log_count / {max_log_obis}) * (log_count / {max_log_obis})) * 215, 
        180
    ]""",
    get_elevation="log_count",
    # Elevation etwas höher als Fischerei (50.000), falls du sie später kombinierst
    elevation_scale=50000, 
    extruded=True,
    pickable=True,
    id="obis-layer"
)

# 5. Kamera-Einstellungen
view_state = pdk.ViewState(
    latitude=20, 
    longitude=0, 
    zoom=2.2, 
    pitch=45
)

# 6. Deck erstellen & als HTML speichern
r = pdk.Deck(
    layers=[layer_obis], 
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
    tooltip={
        "html": "<b>H3 Index:</b> {h3}<br/><b>Sichtungen:</b> {sighting_count}",
        "style": {"color": "white", "backgroundColor": "#004444"}
    }
)

# Speicherpfad
output_file = "views/obis.html"
r.to_html(output_file)

print(f"Die OBIS-Karte wurde erfolgreich unter '{output_file}' erstellt!")

Aggregiere OBIS-Daten auf H3-Resolution 4...
Erfolg! 71,554 Hexagone mit Tiersichtungen aggregiert.
Die OBIS-Karte wurde erfolgreich unter 'views/obis.html' erstellt!


In [29]:
con = duckdb.connect()
con.execute("INSTALL h3 FROM community;")
con.execute("LOAD h3;")

RESOLUTION = 4

# --- 1. AGGREGATION FISCHEREI (ROT) ---
print("Aggregiere Fischereidaten...")
query_gfw = f"""
    SELECT h3_h3_to_string(h3_cell_to_parent(h3_string_to_h3(h3_index), {RESOLUTION})) as h3, 
           SUM(fishing_hours) as val
    FROM read_parquet('/mnt/shared_data/finflow/gfw_clean/*.parquet')
    GROUP BY 1 HAVING val > 0
"""
df_gfw = con.execute(query_gfw).df()
df_gfw['log_val'] = np.log1p(df_gfw['val'])
max_gfw = df_gfw['log_val'].max()

# --- 2. AGGREGATION OBIS (TÜRKIS) ---
print("Aggregiere Tierdaten...")
query_obis = f"""
    SELECT h3_h3_to_string(h3_cell_to_parent(h3_string_to_h3(h3_index), {RESOLUTION})) as h3, 
           COUNT(*) as sighting_count
    FROM read_parquet('/mnt/shared_data/finflow/obis_clean/*.parquet')
    GROUP BY 1 HAVING sighting_count > 0
"""
df_obis = con.execute(query_obis).df()
df_obis['log_count'] = np.log1p(df_obis['sighting_count'])
max_obis = df_obis['log_count'].max()

# --- 3. LAYER DEFINITION ---

# FISCHEREI-LAYER: Basis-Teppich in Rot
fishing_layer = pdk.Layer(
    "H3HexagonLayer",
    df_gfw,
    get_hexagon="h3",
    get_fill_color=f"[40 + ((log_val / {max_gfw}) * (log_val / {max_gfw}) * (log_val / {max_gfw})) * 215, 0, 0, 140]",
    get_elevation="log_val",
    elevation_scale=30000, # Etwas flacher als Basis
    extruded=True,
    pickable=True,
    id="fishing"
)

# ANIMAL-LAYER: Schwebende Hotspots in Türkis
animal_layer = pdk.Layer(
    "H3HexagonLayer",
    df_obis,
    get_hexagon="h3",
    # Türkis: [0, G, B, Alpha]
    get_fill_color=f"[0, 40 + ((log_count / {max_obis}) * (log_count / {max_obis})) * 215, 40 + ((log_count / {max_obis}) * (log_count / {max_obis})) * 215, 200]",
    get_elevation="log_count",
    elevation_scale=60000, # Doppelt so hoch, damit Tiere über der Fischerei "rausschauen"
    extruded=True,
    pickable=True,
    id="animals"
)

# --- 4. MAP & EXPORT ---

view_state = pdk.ViewState(latitude=15, longitude=0, zoom=2.2, pitch=45)

r = pdk.Deck(
    layers=[fishing_layer, animal_layer],
    initial_view_state=view_state,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
    tooltip={
        "html": "<b>Typ:</b> {id}<br/><b>Wert:</b> {val}{sighting_count}",
        "style": {"color": "white", "backgroundColor": "#111"}
    }
)

r.to_html("views/combined_ocean_map.html")
print("Kombinierte Karte erstellt: views/combined_ocean_map.html")

Aggregiere Fischereidaten...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregiere Tierdaten...
Kombinierte Karte erstellt: views/combined_ocean_map.html
